# Web Scraping and Data Cleaning of Tour de France Rider Histor

## Introduction

In this notebook, I will collect, clean, and prepare data related to the nationalities of Tour de France winners. The focus is on the question:  
**"Which nationality has the most unique Tour de France winners?"**

The primary source will be the table _"Tour de France general classification winners by nationality"_ found on Wikipedia – List of Tour de France general classification winners.

The aim is to scrape this HTML table, restructure it into a clean format suitable for further analysis, and export it to a CSV file for use in subsequent analytical notebooks.


## Libraries and Tools Used

The following Python libraries will be used and should be installed before running the notebook:

- `requests`: To send HTTP requests and retrieve HTML from the web  
- `BeautifulSoup` (from `bs4`): To parse HTML content and extract table data  
- `csv`: To write the cleaned data into a CSV file  
- `json`: To write the cleaned data into a json file
- `re`: Regular expressions for data cleaning if needed


In [75]:
# !pip install requests
# !pip install beautifulsoup4

In [76]:
import requests
from bs4 import BeautifulSoup
import csv
import json
import re

## `tdf_winner_nationalities_clean`



This dataset contains clean, structured data describing the nationalities of Tour de France general classification winners, as obtained by scraping the “By nationality” table from Wikipedia – List of Tour de France general classification winners.

Each record provides:

- The country  
- The total number of Tour de France wins associated with that country  
- A list of winning cyclists from that country

The purpose of building and saving this file is to quickly and reproducibly answer:  
**“Which nationality has the most unique Tour de France winners?”**

In [77]:
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners#By_nationality"

# Headers to mimic a real browser visit 
headers = {
    "User-Agent": "Mozilla/5.0"
}

page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

div = soup.find("div",{"class":"mw-content-ltr mw-parser-output"})
# print(div.prettify())

table_nationality = div.find_all("table", {"class":"wikitable"})[3]
# print(table_nationality.prettify())

rows = table_nationality.find_all("tr")[1:]  # Skip header row

# Extract data
data = []
for row in rows:
    country = row.find("th")  
    wins = row.find_all("td")[0]
    winning_cyclist = row.find_all("td")[1]
    # print(country, wins, winning_cyclist)
    data.append([country.text.strip(), wins.text.strip(), winning_cyclist.text.strip()])    # Strip whitespace and newlines
# print(data)

# Export to CSV
header = ["Country", "Winners", "Winning_Cyclists"]
with open('data/tdf_winner_nationalities_clean.csv','w',encoding='utf-8',newline='') as f:
    writer = csv.writer(f) # f is the file object and we pass it to the csv.writer
    writer.writerow(header) # write the header (first row)
    writer.writerows(data) # write all data rows (data is a list of lists)

## `tdf_multiple_winners_clean.json`

This dataset contains clean, structured information on cyclists who have won the Tour de France more than three times, as obtained by scraping the “Multiple winners of the Tour de France general classification” table from Wikipedia – List of Tour de France general classification winners.

Each record provides:

- The cyclist’s name  
- The cyclist’s surname  
- The total number of Tour de France wins by that cyclist  
- The specific years in which each win was achieved  
- The cyclist’s nationality, if available
- The link to the year of the win on Wikipedia

The purpose of building and saving this file is to enable a fast and reproducible answer to:  
**“Which cyclists have won the Tour more than three times, and in which years?”**

All cleaned data is stored as a structured JSON file:

`data/tdf_multiple_winners_clean.json`

This format is optimal for semi-structured records like name, win count, and list of years, and will be easy to reload for future analyses or sharing.

In [ ]:
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners#By_nationality"

# Headers to mimic a real browser visit 
headers = {
    "User-Agent": "Mozilla/5.0"
}

page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

div = soup.find("div",{"class":"mw-content-ltr mw-parser-output"})

table_multiple_winners = div.find_all("table", {"class":"wikitable"})[2]
rows = table_multiple_winners.find_all("tr")[1:]  # Skip header row

cyclist_data = []

for row in rows:
    cols = row.find_all("td")

    cyclist_surname = re.search(r'-value="(.+), ',str(row))
    cyclist_name = re.search(r', (\w+)"',str(row))
    country = re.search(r'<abbr title="(.+)"',str(row))

    wins_td = cols[-1]
    len_wins = len(wins_td.find_all("a"))
    years = [a.text for a in wins_td.find_all("a")]
    links_years = [f"https://en.wikipedia.org{a['href']}" for a in wins_td.find_all("a")]

    wins_years = [{"year": y, "link": l} for y, l in zip(years, links_years)]
    
    cyclist_data.append({
        "surname": cyclist_surname.group(1),
        "name": cyclist_name.group(1),
        "country": country.group(1) if country else None,
        "win_count": len_wins,
        "wins_years": wins_years
    })

# Export to JSON
with open('data/tdf_multiple_winners_clean.json', 'w', encoding='utf-8') as f:
    json.dump(cyclist_data, f, indent=3)
